In [1]:
# 詳細な診断情報を収集
import sys
import subprocess
import os

print("=== Python環境情報 ===")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Python path: {sys.path}")

print("\n=== pip情報 ===")
pip_result = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True)
print(f"pip version: {pip_result.stdout.strip()}")

print("\n=== requirements.txtの確認 ===")
requirements_path = "/workspace/requirements.txt"
if os.path.exists(requirements_path):
    with open(requirements_path, 'r') as f:
        content = f.read()
    print(f"requirements.txt found at: {requirements_path}")
    print("Content:")
    print(content)
else:
    print("requirements.txt not found in /workspace")
    # 他の場所も確認
    for path in ["./requirements.txt", "/requirements.txt", "/tmp/requirements.txt"]:
        if os.path.exists(path):
            print(f"Found requirements.txt at: {path}")

print("\n=== インストール済みパッケージ ===")
result = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True)
print(result.stdout)

# ociパッケージの確認
if 'oci' in result.stdout.lower():
    print("\n✓ OCI package is installed")
    try:
        import oci
        print(f"OCI SDK version: {oci.__version__}")
    except ImportError as e:
        print(f"OCI import error: {e}")
else:
    print("\n✗ OCI package is NOT installed")

=== Python環境情報 ===
Python executable: /usr/local/bin/python
Python version: 3.11.13 (main, Aug 14 2025, 07:04:20) [GCC 14.2.0]
Python path: ['/usr/local/lib/python311.zip', '/usr/local/lib/python3.11', '/usr/local/lib/python3.11/lib-dynload', '', '/home/vscode/.local/lib/python3.11/site-packages', '/usr/local/lib/python3.11/site-packages']

=== pip情報 ===
pip version: pip 25.2 from /usr/local/lib/python3.11/site-packages/pip (python 3.11)

=== requirements.txtの確認 ===
requirements.txt found at: /workspace/requirements.txt
Content:
oci

=== インストール済みパッケージ ===
Package                   Version
------------------------- --------------
anyio                     4.11.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.5
attrs                     25.3.0
babel                     2.17.0
beautifulsoup4            4.14.2
bleach                    6.2.0
certifi                   2025.8.3
cf

In [17]:
# OCI GenAI サービスの設定と認証情報の確認
import oci
import os
from oci.generative_ai_inference import GenerativeAiInferenceClient
from oci.generative_ai_inference.models import GenerateTextDetails, CohereLlmInferenceRequest

# OCI設定の確認
print("OCI SDK version:", oci.__version__)

# 設定ファイルまたは環境変数から認証情報を読み込み
try:
    # デフォルトプロファイルを使用
    config = oci.config.from_file()
    print("OCI config loaded successfully")
except Exception as e:
    print(f"Config loading error: {e}")
    print("環境変数またはinstance principalを使用してください")

OCI SDK version: 2.160.3
OCI config loaded successfully


# GenAIでプロンプトに応答

In [ ]:
# OCI GenAI チャットモデルクライアント（修正版）
import oci
import os
from oci.generative_ai_inference import GenerativeAiInferenceClient
from oci.generative_ai_inference.models import ChatDetails, CohereChatRequest, CohereMessage

class OCIGenAIChatClient:
    def __init__(self, compartment_id, endpoint_url=None, model_id="cohere.command-r-plus"):
        """
        OCI GenAI チャットクライアントの初期化
        
        Args:
            compartment_id (str): OCI コンパートメントID
            endpoint_url (str): GenAI サービスのエンドポイントURL
            model_id (str): 使用するチャットモデルID
        """
        self.compartment_id = compartment_id
        self.model_id = model_id
        self.config = oci.config.from_file()
        
        # リージョンに基づいてエンドポイントを設定
        if endpoint_url is None:
            region = self.config.get('region', 'us-chicago-1')
            self.endpoint_url = f"https://inference.generativeai.{region}.oci.oraclecloud.com"
        else:
            self.endpoint_url = endpoint_url
            
        self.client = GenerativeAiInferenceClient(
            config=self.config,
            service_endpoint=self.endpoint_url
        )
        print(f"GenAI Chat client initialized for region: {self.config.get('region')}")
        print(f"Using model: {self.model_id}")
    
    def chat(self, messages, max_tokens=500, temperature=0.7, top_p=0.95, top_k=0):
        """
        チャット形式でのテキスト生成
        
        Args:
            messages (list): 会話履歴 [{"role": "USER"|"CHATBOT"|"SYSTEM", "message": "..."}]
            max_tokens (int): 最大トークン数
            temperature (float): 温度パラメータ (0.0-1.0)
            top_p (float): Top-p サンプリング (0.0-1.0)
            top_k (int): Top-k サンプリング
        
        Returns:
            dict: 生成結果とメタデータ
        """
        try:
            # メッセージを CohereMessage オブジェクトに変換
            cohere_messages = []
            for msg in messages:
                role = msg.get("role", "USER").upper()
                content = msg.get("message", msg.get("content", ""))
                
                # ロールの正規化
                if role in ["USER", "HUMAN"]:
                    role = "USER"
                elif role in ["ASSISTANT", "CHATBOT", "BOT"]:
                    role = "CHATBOT"
                elif role == "SYSTEM":
                    role = "SYSTEM"
                
                cohere_messages.append(CohereMessage(
                    role=role,
                    message=content
                ))
            
            # 現在のメッセージと履歴を分離
            current_message = cohere_messages[-1].message if cohere_messages else ""
            chat_history = cohere_messages[:-1] if len(cohere_messages) > 1 else []
            
            # Cohere Chat リクエストの作成
            cohere_chat_request = CohereChatRequest(
                message=current_message,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                frequency_penalty=0.0,
                presence_penalty=0.0,
                chat_history=chat_history
            )
            
            # チャットリクエストの詳細設定
            chat_details = ChatDetails(
                compartment_id=self.compartment_id,
                serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(
                    model_id=self.model_id
                ),
                chat_request=cohere_chat_request
            )
            
            # API呼び出し実行
            print(f"Generating chat response...")
            response = self.client.chat(chat_details)
            
            # 生成されたテキストの抽出
            generated_text = response.data.chat_response.text
            
            return {
                "success": True,
                "response": generated_text,
                "model_id": self.model_id,
                "parameters": {
                    "max_tokens": max_tokens,
                    "temperature": temperature,
                    "top_p": top_p,
                    "top_k": top_k
                },
                "message_count": len(messages)
            }
            
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "messages": messages
            }
    
    def simple_chat(self, user_message, chat_history=None, system_prompt=None):
        """
        シンプルなチャット関数
        
        Args:
            user_message (str): ユーザーのメッセージ
            chat_history (list, optional): 過去の会話履歴
            system_prompt (str, optional): システムプロンプト
        
        Returns:
            str: AIの応答
        """
        messages = []
        
        # システムプロンプトの追加
        if system_prompt:
            messages.append({
                "role": "SYSTEM",
                "message": system_prompt
            })
        
        # 会話履歴の追加
        if chat_history:
            messages.extend(chat_history)
        
        # 現在のユーザーメッセージを追加
        messages.append({
            "role": "USER",
            "message": user_message
        })
        
        result = self.chat(messages)
        return result["response"] if result["success"] else f"Error: {result['error']}"

# 使用例とテスト
COMPARTMENT_ID = "ocid1.compartment.oc1..aaaaaaaanxm4oucgt5pkgd7sw2vouvckvvxan7ca2lirowaao7krnzlkdkhq"

try:
    # チャットクライアントの初期化
    chat_client = OCIGenAIChatClient(COMPARTMENT_ID)
    
    print("=== OCI GenAI チャットモデル テスト ===\n")
    
    # シンプルなチャットテスト
    print("1. シンプルなチャットテスト:")
    user_msg = "Oracle Cloud Infrastructure GenAIサービスについて教えてください。"
    response = chat_client.simple_chat(user_msg)
    print(f"ユーザー: {user_msg}")
    print(f"AI: {response}\n")
    
    # 会話履歴付きチャットテスト
    print("2. 会話履歴付きチャットテスト:")
    conversation = [
        {"role": "USER", "message": "こんにちは！"},
        {"role": "CHATBOT", "message": "こんにちは！今日はどのようなお手伝いができますか？"},
        {"role": "USER", "message": "OCIのGenAIサービスについて教えてください"}
    ]
    
    result = chat_client.chat(conversation)
    if result["success"]:
        print("会話履歴:")
        for msg in conversation:
            role_name = "ユーザー" if msg["role"] == "USER" else "AI"
            print(f"{role_name}: {msg['message']}")
        print(f"AI: {result['response']}")
        print(f"メタデータ: {result['parameters']}\n")
    
    # システムプロンプト付きチャット
    print("3. システムプロンプト付きチャット:")
    system_prompt = "あなたは親切で知識豊富なAIアシスタントです。日本語で丁寧に回答してください。"
    user_question = "Pythonでリストを辞書に変換する方法は？"
    
    response = chat_client.simple_chat(
        user_message=user_question,
        system_prompt=system_prompt
    )
    print(f"システムプロンプト: {system_prompt}")
    print(f"ユーザー: {user_question}")
    print(f"AI: {response}")
    
except Exception as e:
    print(f"初期化エラー: {e}")
    print("\n以下を確認してください:")
    print("1. COMPARTMENT_ID が正しく設定されているか")
    print("2. ~/.oci/config ファイルが存在し、正しく設定されているか")
    print("3. OCI GenAI サービスがお使いのリージョンで利用可能か")
    print("4. 必要な権限が設定されているか")
    print("5. チャットモデル（cohere.command-r-plus等）が利用可能か")

SyntaxError: keyword argument repeated: message (1018190554.py, line 80)

In [19]:
tmp = oci.generative_ai.GenerativeAiClient(config)

In [20]:
tmp.list_models(COMPARTMENT_ID).data

{
  "items": [
    {
      "base_model_id": null,
      "capabilities": [
        "UNKNOWN_ENUM_VALUE"
      ],
      "compartment_id": null,
      "defined_tags": {},
      "display_name": "meta.llama-guard-4-12b",
      "fine_tune_details": null,
      "freeform_tags": {},
      "id": "ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyaf4q5ji7iw7k3h6ol4a6lgpk7jnjuc5xlq55z4kxyfecq",
      "is_long_term_supported": true,
      "lifecycle_details": "Creating Base Model",
      "lifecycle_state": "ACTIVE",
      "model_metrics": null,
      "system_tags": {},
      "time_created": "2025-08-19T19:44:09.634000+00:00",
      "time_dedicated_retired": null,
      "time_deprecated": "2025-08-01T00:00:00+00:00",
      "time_on_demand_retired": null,
      "type": "BASE",
      "vendor": "meta",
      "version": "1.0.0"
    },
    {
      "base_model_id": null,
      "capabilities": [
        "CHAT"
      ],
      "compartment_id": null,
      "defined_tags": {},
      "display_name": "op